# Ablation Study: Text Preprocessing Parameters

This notebook systematically evaluates the impact of three preprocessing toggles — **normalisation** (`norm`), **numeric removal** (`num`), and **stop-word removal** (`stop`) — across both pipelines.

To ensure a fair comparison between vectorisation strategies, **both classifiers** (LinearSVC and Random Forest) are tested on **both pipelines**:

| Pipeline | Cleaner | Vectoriser | Classifiers Tested |
|----------|---------|------------|-------------------|
| BOW | `StemCleaner` | TF-IDF char n-grams | LinearSVC, Random Forest |
| Semantic | `LemmaCleaner` | Sentence-BERT embeddings | LinearSVC, Random Forest |

Each pipeline × classifier combination is run under all 8 combinations of the three binary flags. We report **5-fold CV macro-F1** (on the training set) and **test-set macro-F1** for every configuration. The classifier that performs best across both pipelines is then selected as the **unified model**.

In [1]:
from datasets import load_dataset

import itertools
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

from Cleaners import StemCleaner, LemmaCleaner
warnings.filterwarnings('ignore')

In [2]:
raw = load_dataset("csv", data_files="cw_data.csv", column_names=["text", "label"])
dataset = raw["train"]
y = dataset["label"]

## 1 Cleaner Definitions

Identical to the classes in `BOW_Approach.ipynb` and `Semantic_Approach.ipynb`.

## 2 Ablation Runner

We iterate over every `(norm, num, stop)` combination for both pipelines, testing **both classifiers** (LinearSVC and Random Forest) on each pipeline to determine which classifier works best as a unified model.

In [ ]:
FLAG_COMBOS = list(itertools.product([False, True], repeat=3))

sbert = SentenceTransformer('all-MiniLM-L6-v2')

classifiers = {
    'LinearSVC': lambda: LinearSVC(class_weight='balanced', random_state=42),
    'RandomForest': lambda: RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
}

results = []

for norm, num, stop in FLAG_COMBOS:
    tag = f"norm={norm}, num={num}, stop={stop}"
    print(f"Running: {tag}")

    # ── BOW preprocessing + vectorisation ──
    parsed_bow = StemCleaner(dataset["text"], norm=norm, num=num, stop=stop)
    train_t, test_t, y_train, y_test = train_test_split(
        parsed_bow.rejoined, y, test_size=0.2, random_state=42, stratify=y
    )
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
    X_train_bow = vec.fit_transform(train_t)
    X_test_bow  = vec.transform(test_t)

    # ── Semantic preprocessing + vectorisation ──
    parsed_sem = LemmaCleaner(dataset["text"], norm=norm, num=num, stop=stop)
    embeddings = sbert.encode(parsed_sem.rejoined)
    X_train_sem, X_test_sem, y_train_sem, y_test_sem = train_test_split(
        embeddings, y, test_size=0.2, random_state=42, stratify=y
    )

    # ── Test both classifiers on both pipelines ──
    for clf_name, clf_fn in classifiers.items():
        # BOW + classifier
        bow_model = clf_fn()
        bow_model.fit(X_train_bow, y_train)
        bow_cv = cross_val_score(bow_model, X_train_bow, y_train, cv=5, scoring='f1_macro')
        bow_test_f1 = f1_score(y_test, bow_model.predict(X_test_bow), average='macro')

        results.append({
            'pipeline': 'BOW', 'classifier': clf_name,
            'norm': norm, 'num': num, 'stop': stop,
            'cv_f1_mean': bow_cv.mean(), 'cv_f1_std': bow_cv.std(),
            'test_f1': bow_test_f1,
        })

        # Semantic + classifier
        sem_model = clf_fn()
        sem_model.fit(X_train_sem, y_train_sem)
        sem_cv = cross_val_score(sem_model, X_train_sem, y_train_sem, cv=5, scoring='f1_macro')
        sem_test_f1 = f1_score(y_test_sem, sem_model.predict(X_test_sem), average='macro')

        results.append({
            'pipeline': 'Semantic', 'classifier': clf_name,
            'norm': norm, 'num': num, 'stop': stop,
            'cv_f1_mean': sem_cv.mean(), 'cv_f1_std': sem_cv.std(),
            'test_f1': sem_test_f1,
        })

df = pd.DataFrame(results)
print(f"\nDone — collected {len(df)} rows.")

## 3 Results Table

### 3.1 Full Results (all pipeline × classifier × flag combinations)

In [ ]:
display_df = df.copy()
display_df['CV F1'] = display_df.apply(
    lambda r: f"{r['cv_f1_mean']:.3f} ± {r['cv_f1_std']:.3f}", axis=1
)
display_df['Test F1'] = display_df['test_f1'].map('{:.3f}'.format)
display_df['Pipeline + Classifier'] = display_df['pipeline'] + ' + ' + display_df['classifier']

display_df[['Pipeline + Classifier', 'norm', 'num', 'stop', 'CV F1', 'Test F1']]

### 3.2 Classifier Comparison Summary

Average test macro-F1 for each classifier across all 8 flag configurations, per pipeline.

In [ ]:
clf_summary = df.groupby(['pipeline', 'classifier']).agg(
    mean_test_f1=('test_f1', 'mean'),
    std_test_f1=('test_f1', 'std'),
    mean_cv_f1=('cv_f1_mean', 'mean'),
).round(4)

print("=== Classifier Performance Summary ===\n")
print(clf_summary.to_string())

print("\n\n=== Average Test F1 across BOTH pipelines (unified suitability) ===\n")
unified = df.groupby('classifier')['test_f1'].mean().round(4)
print(unified.to_string())

best_clf = unified.idxmax()
print(f"\n→ Best unified classifier: {best_clf} (avg test F1 = {unified[best_clf]:.4f})")

## 4 Per-Parameter Effect (Marginal Analysis)

For each flag we compute the **average test macro-F1 when the flag is ON vs OFF**, marginalised over the other two flags. This isolates the individual contribution of each preprocessing step. Results are shown per pipeline × classifier combination.

In [ ]:
marginal_rows = []
for (pipe, clf), sub in df.groupby(['pipeline', 'classifier']):
    for flag in ['norm', 'num', 'stop']:
        off = sub[sub[flag] == False]['test_f1'].mean()
        on  = sub[sub[flag] == True]['test_f1'].mean()
        marginal_rows.append({
            'pipeline': pipe, 'classifier': clf, 'parameter': flag,
            'OFF': round(off, 4), 'ON': round(on, 4), 'delta': round(on - off, 4)
        })

marginal = pd.DataFrame(marginal_rows)
marginal['delta_fmt'] = marginal['delta'].map(lambda d: f"{d:+.4f}")
marginal[['pipeline', 'classifier', 'parameter', 'OFF', 'ON', 'delta_fmt']]

## 5 Visualisation

### 5.1 Grouped Bar Chart — All 8 Configurations per Pipeline × Classifier

In [ ]:
import numpy as np

combos = [
    ('BOW', 'LinearSVC'),
    ('BOW', 'RandomForest'),
    ('Semantic', 'LinearSVC'),
    ('Semantic', 'RandomForest'),
]
colors = ['#4C72B0', '#7EAED2', '#DD8452', '#F2C68A']

labels = [
    f"{'N' if r.norm else '-'}{'U' if r.num else '-'}{'S' if r.stop else '-'}"
    for _, r in df[(df['pipeline'] == 'BOW') & (df['classifier'] == 'LinearSVC')].iterrows()
]

fig, ax = plt.subplots(figsize=(14, 5))
n_groups = len(labels)
n_bars = len(combos)
w = 0.8 / n_bars
x = np.arange(n_groups)

for i, (pipe, clf) in enumerate(combos):
    sub = df[(df['pipeline'] == pipe) & (df['classifier'] == clf)].reset_index(drop=True)
    ax.bar(x + i * w - (n_bars - 1) * w / 2, sub['test_f1'], w,
           label=f'{pipe} + {clf}', color=colors[i])

ax.set_xlabel('Flag combination  (N=norm, U=num, S=stop;  "-" = off)')
ax.set_ylabel('Test Macro-F1')
ax.set_title('Ablation: Test Macro-F1 across all preprocessing configurations')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.legend(fontsize=9)
ax.set_ylim(bottom=max(0, df['test_f1'].min() - 0.05))
plt.tight_layout()
plt.show()

### 5.2 Marginal Delta Plot — Per-Parameter Impact by Pipeline × Classifier

In [ ]:
params = ['norm', 'num', 'stop']
combos = [
    ('BOW', 'LinearSVC'),
    ('BOW', 'RandomForest'),
    ('Semantic', 'LinearSVC'),
    ('Semantic', 'RandomForest'),
]
colors = ['#4C72B0', '#7EAED2', '#DD8452', '#F2C68A']

fig, ax = plt.subplots(figsize=(10, 5))
n_params = len(params)
n_bars = len(combos)
w = 0.8 / n_bars
x = np.arange(n_params)

for i, (pipe, clf) in enumerate(combos):
    sub = marginal[(marginal['pipeline'] == pipe) & (marginal['classifier'] == clf)]
    deltas = sub.set_index('parameter').loc[params, 'delta']
    ax.bar(x + i * w - (n_bars - 1) * w / 2, deltas, w,
           label=f'{pipe} + {clf}', color=colors[i])

ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(params)
ax.set_ylabel('Δ Test Macro-F1  (ON − OFF)')
ax.set_title('Marginal effect of each preprocessing flag')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6 Best Configuration per Pipeline × Classifier

In [ ]:
for (pipe, clf), sub in df.groupby(['pipeline', 'classifier']):
    best = sub.loc[sub['test_f1'].idxmax()]
    print(f"{pipe} + {clf}")
    print(f"  Best config : norm={best['norm']}, num={best['num']}, stop={best['stop']}")
    print(f"  Test F1     : {best['test_f1']:.3f}")
    print(f"  CV F1       : {best['cv_f1_mean']:.3f} ± {best['cv_f1_std']:.3f}")
    print()

## 7 Discussion

### Classifier Selection for Unified Model

To ensure a fair comparison between the BOW and Semantic vectorisation strategies, both pipelines must use the **same classifier**. We evaluated LinearSVC and Random Forest on both pipelines across all 8 preprocessing configurations.

The classifier comparison summary (Section 3.2) shows the average test F1 for each classifier across both pipelines. The classifier with the higher combined average is selected as the **unified model** for the final ablation comparison.

### Preprocessing Flag Effects

**`norm` (text normalisation)** — Maps informal tokens (`wat` → `what`, `acct` → `account`, `cant` → `cannot`) to their standard forms. This is expected to help the BOW pipeline more than the semantic pipeline, because character n-gram TF-IDF cannot infer that `wat` and `what` share meaning, whereas Sentence-BERT embeddings can often capture informal synonyms from pre-training context.

**`num` (numeric removal)** — Strips standalone digits like `2`. Since the normalisation map already converts `2` → `to` when `norm=True`, this flag's effect is mostly visible when `norm=False`. Removing numerics can reduce noise but may also discard meaningful tokens in contexts like "2-factor".

**`stop` (stop-word removal)** — Removes high-frequency function words while preserving negations (`not`, `no`, `cannot`). For the BOW pipeline this concentrates TF-IDF weight on content words. For the semantic pipeline, stop words carry syntactic context that Sentence-BERT was trained on, so removing them may hurt embedding quality.

### Key Takeaway

By using the same classifier on both pipelines, any performance difference is attributable **solely to the vectorisation strategy** (TF-IDF char n-grams vs Sentence-BERT embeddings) and the associated preprocessing (stemming vs lemmatisation). This controlled comparison provides a cleaner answer to the central research question: does semantic vectorisation outperform bag-of-words for this intent classification task?